# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/madaraf/Starter-Notebooks-Assignment-Flyrank-/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

# Question 1 — What does one row mean?
One row = one content page, summarized over a time window. Not one page-per-day (which is the warehouse's raw grain), and not one client — a single page gets aggregated down to exactly one row, the same shape my Week 1-2 work already used, just now built from the real warehouse data instead of the pre-made starter CSV.
# Question 2 — Which table(s) will I use?
**fact_content_daily_performance** (for the numeric activity signals — impressions, clicks, position, sessions) joined to **dim_content** (for content metadata — content type, word count, main intent). I'm not using **fact_content_query_90d** yet — that's query-mix data I don't need for a first 5-feature pass, and skipping it keeps my joins simple this week.

# Question 3 — Which time window?
A real forward-looking split: prior 90 days as my feature window, and a separate, later 30-day window as my outcome window — features from the past, label from the future, with a gap between them. I'll develop this on one mid-panel month (e.g. **month=2026-03**), never the **_sample** table, since **_sample** is the panel's last month and using it here would mean peeking at the future I'm trying to predict.
# Question 4 — What would I predict or rank? (label or proxy)
Whether a page's impressions drop 20%+ from the feature window into the outcome window — same "declining" idea as before, but now genuinely forward-looking instead of measured inside one window. I'm adding a minimum volume floor (**impressions in the feature window >= 100**) so tiny, noisy pages don't get flagged as "declining" just because their small numbers wobbled.
# Question 5 — What do I deliberately exclude, and why?
Two things:


1.   **trend_direction** / **trend_pct** (and their warehouse equivalents) — excluded as features, because my label is built from this exact signal. Including them would mean the model just reads the answer instead of learning anything (this is the leakage trap I'll deliberately demonstrate, then remove, in Task 4).
2.   **client_id** — excluded as a feature. It's a pseudonymous key, useful only for grouping/joining/splitting (e.g. a client-holdout split later), never something the model should learn a pattern from directly.



In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

**Feature** — knowable inside my prior-90-day feature window, before the decision moment:

| Field | Why it's safe |
|---|---|
| `gsc_impressions`, `gsc_clicks` (summed over the **feature window only**) | Real search activity that already happened before prediction time |
| `gsc_avg_position` (averaged over the feature window) | Search ranking position, already observed by the time I'd predict |
| GA4 sessions / engaged_sessions / scroll_events (feature window only) | Post-click behavior already measured before prediction time |
| `content_type`, `main_intent`, `word_count` (from `dim_content`) | Static-ish content metadata, doesn't move based on the outcome |
| Content age (derived from `content_created_at`) | Known the moment the page exists, not tied to future traffic |

**Label / proxy** — the thing I'm predicting, built from the separate, later outcome window:

| Field | Role |
|---|---|
| `gsc_impressions` summed over the **outcome window only** | Compared against the feature-window total to compute my forward-looking decline flag (>=20% drop, with a minimum-volume floor) |

**Context** — for joining, grouping, or splitting only; never fed to the model as a learned input:

| Field | Why |
|---|---|
| `client_hash_id` | Used for a client-holdout split later, never a learned input |
| `content_hash_id` | Row identity / join key only |
| `url_hash_id`, `keyword_hash_id` | Pseudonymous grouping/dedup keys — never mapped back, never a feature |
| `report_date` | Used to define the two time windows, not a per-row input value itself |

**Excluded** — deliberately not used, each with a reason:

| Field | Why excluded |
|---|---|
| `gsc_impressions` / `gsc_clicks` from the **outcome window**, used *as a feature* | The label-derived leakage trap: same column as my feature, but summed over the window I'm trying to predict — using it as an input would mean reading the answer key instead of learning a pattern (demonstrated on purpose, then removed, in Task 4) |
| `provider_used`, `model_used` | Flagged in the data dictionary as "not a model feature" — describes how content was generated, not how it performs |
| `fact_content_query_90d` (whole table) | Skipped this week per my Question 2 decision — its 90-day window overlaps recent months and risks silently leaking into a forward-looking label; not worth the risk for a first 5-feature pass |

**Note on the repeated column:** `gsc_impressions`/`gsc_clicks` appear in three buckets on purpose — same column name, but the date range it's summed over decides its job. Feature = summed over the *earlier* window. Label = summed over the *later* window. Excluded = the later-window version, if it were mistakenly reused as an input.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Every claim in Sections 1-2 above gets checked against real warehouse data below. I'm developing on one mid-panel month (**month=2026-03**) — never the **_sample** table, since that's the panel's last month and I need genuine "before/after" room for my forward-looking label later. Three checks: does the grain really match what I claimed, how big is this slice and what dates does it span, and how many rows actually have usable GA4 data once I filter correctly.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

Paste your Hugging Face READ token (hf_...): ··········


Fact 1 — The grain. I claimed one row = one content page, per day (before I aggregate it myself). If that's true, grouping by `report_date + client_hash_id + content_hash_id` should never find more than one row per combination. Zero rows back proves the grain holds.

In [4]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS row_count
    FROM {MONTH}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"Rows violating the claimed grain: {len(grain_check)}")
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating the claimed grain: 0


,report_date,client_hash_id,content_hash_id,row_count


Fact 2 — Row count and date span. I claimed this is one mid-panel month. Let's see exactly how many rows that is and confirm the dates really only cover March 2026 (not accidentally spilling into other months).

In [5]:
month_shape = con.sql(f"""
    SELECT
        COUNT(*)                       AS total_rows,
        COUNT(DISTINCT content_hash_id) AS unique_content,
        COUNT(DISTINCT client_hash_id)  AS unique_clients,
        MIN(report_date)               AS earliest_date,
        MAX(report_date)               AS latest_date
    FROM {MONTH}
""").df()

month_shape

,total_rows,unique_content,unique_clients,earliest_date,latest_date
0,9841378,331437,55,2026-03-01,2026-03-31


What I'm checking: **earliest_date/latest_date** should both fall inside March 2026 — if they don't, my partition filter isn't doing what I think it's doing, and I need to fix the path before trusting anything downstream.

Fact 3 — Availability, filtered correctly with **IS TRUE**. Per the data dictionary's warning: **ga4_data_available** isn't just TRUE/FALSE — it can also be NULL (rows where the flag itself is missing, not just "no GA4 yet"). Using **= FALSE** or NOT **ga4_data_availabl**e would silently mishandle those NULLs. I filter with **IS TRUE / IS NOT TRUE** specifically to see how many rows actually survive a correct filter versus a naive one.

In [6]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)     AS ga4_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS ga4_unavailable_or_null_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS NULL)     AS ga4_flag_is_null_rows
    FROM {MONTH}
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,ga4_unavailable_or_null_rows,ga4_flag_is_null_rows
0,9841378,413966,9427412,3018741


### 3. Five Features
**The Feature Contract:**
1. **impressions_month** — total GSC impressions summed over March. Knowable because it's a plain count of search events that already happened by the end of the month — nothing about it depends on what happens in April.
2. **ctr_month** — clicks ÷ impressions for March. Knowable because both clicks and impressions are already-observed March activity; the ratio doesn't reference anything outside this window.
3. **avg_position_month** — the page's average search ranking position across March. Knowable because Google already assigned this position during the month; it's a record of where the page already sat in results, not a future outcome.
4. **days_with_impressions** — how many distinct days in March the page got at least one impression. Knowable because it's just counting activity that already happened — a measure of how consistently visible the page was, not how it will perform next..
5. **content_age_days** — days between the page's creation date and the end of March. Knowable because the page's creation date is fixed and already in the past; age never depends on future traffic.

**5 features, built from the same March 2026 month**. For each one, I ask: "was this knowable before the decision moment?" — meaning, could I have computed this using only data up through the end of my feature window, with zero peeking into what happens after. All five here pass that test; none of them touch **trend_direction/trend_pct** or anything from a later window

In [7]:
features_df = con.sql(f"""
    WITH page_month AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(gsc_impressions)                              AS impressions_month,
            SUM(gsc_clicks)                                   AS clicks_month,
            AVG(gsc_avg_position)                             AS avg_position_month,
            COUNT(*) FILTER (WHERE gsc_impressions > 0)       AS days_with_impressions
        FROM {MONTH}
        GROUP BY content_hash_id, client_hash_id
        HAVING SUM(gsc_impressions) >= 100
    )
    SELECT
        p.content_hash_id,
        p.client_hash_id,
        p.impressions_month,
        p.clicks_month,
        ROUND(p.clicks_month * 1.0 / NULLIF(p.impressions_month, 0), 4) AS ctr_month,
        p.avg_position_month,
        p.days_with_impressions,
        DATE_DIFF('day', d.content_created_date, DATE '2026-03-31')       AS content_age_days
    FROM page_month p
    LEFT JOIN read_parquet('{REL}/dim_content.parquet') d
        ON p.content_hash_id = d.content_hash_id
    LIMIT 200000
""").df()

print(f"Feature frame shape: {features_df.shape}")
features_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (101441, 8)


,content_hash_id,client_hash_id,impressions_month,clicks_month,ctr_month,avg_position_month,days_with_impressions,content_age_days
0,content_d0dff76c889de68f,client_62f4a7e64f5e0096,181.0,0.0,0.0000,5.147402,29,47
1,content_d49a012dcb924e31,client_62f4a7e64f5e0096,329.0,0.0,0.0000,5.177774,31,47
2,content_cec711b02f3bbde6,client_62f4a7e64f5e0096,602.0,4.0,0.0066,4.428747,29,47
3,content_614baf2af4330bd7,client_62f4a7e64f5e0096,772.0,1.0,0.0013,4.685335,31,47
4,content_755d951187fcd70a,client_62f4a7e64f5e0096,1858.0,6.0,0.0032,1.854929,30,47


# The leakage trap

In [8]:
# Step 1: Build the forward-looking label using a SEPARATE, later month (April)
APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"

april_impressions = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_impressions) AS impressions_april
    FROM {APRIL}
    GROUP BY content_hash_id
""").df()

# Join April's outcome onto our March feature frame
labeled_df = features_df.merge(april_impressions, on="content_hash_id", how="inner")

# The label: did impressions drop 20%+ from March into April?
labeled_df["pct_change"] = (
    (labeled_df["impressions_april"] - labeled_df["impressions_month"])
    / labeled_df["impressions_month"]
)
labeled_df["is_declining_forward_label"] = (labeled_df["pct_change"] <= -0.20).astype(int)

print(f"Rows with a real forward label: {len(labeled_df):,}")
print(f"Declining rate: {labeled_df['is_declining_forward_label'].mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows with a real forward label: 101,441
Declining rate: 0.518


In [9]:
# Step 2: Train HONESTLY, using only the 5 legitimate features
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ["impressions_month", "ctr_month", "avg_position_month",
                    "days_with_impressions", "content_age_days"]

X = labeled_df[honest_features].fillna(0)
y = labeled_df["is_declining_forward_label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

honest_model = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
honest_model.fit(X_train, y_train)
honest_score = roc_auc_score(y_test, honest_model.predict_proba(X_test)[:, 1])

print(f"HONEST model ROC AUC (5 legitimate features only): {honest_score:.3f}")

HONEST model ROC AUC (5 legitimate features only): 0.677


In [10]:
# Step 3: Now CHEAT on purpose — sneak in a label-derived column
# pct_change IS literally the exact number the label was computed from — pure leakage.
leaky_features = honest_features + ["pct_change"]

X_leaky = labeled_df[leaky_features].fillna(0)
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leaky, y, test_size=0.25, random_state=42, stratify=y
)

leaky_model = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
leaky_model.fit(X_train_l, y_train_l)
leaky_score = roc_auc_score(y_test_l, leaky_model.predict_proba(X_test_l)[:, 1])

print(f"LEAKY model ROC AUC (5 features + pct_change): {leaky_score:.3f}  <- suspiciously close to 1.0")
print(f"\nJump from leakage: {leaky_score:.3f} - {honest_score:.3f} = {leaky_score - honest_score:.3f}")

LEAKY model ROC AUC (5 features + pct_change): 1.000  <- suspiciously close to 1.0

Jump from leakage: 1.000 - 0.677 = 0.323


In [11]:
# Step 4: Delete the leaky column, keep only the honest result
del leaky_model, X_leaky, leaky_features
print("Leaky feature and leaky model deleted.")
print(f"The number I actually report and trust: HONEST ROC AUC = {honest_score:.3f}")

Leaky feature and leaky model deleted.
The number I actually report and trust: HONEST ROC AUC = 0.677


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Named limitation: the availability flag is unreliable for roughly a third of this month's rows, and it fails silently if you're not careful.

From Fact 3 (Task 2), out of 9,841,378 rows in March 2026: 413,966 have confirmed GA4 data (IS TRUE), but of the remaining rows, 3,018,741 (about 31% of the whole month) have ga4_data_available = NULL — not "confirmed unavailable," but genuinely unknown. A careless filter like WHERE ga4_data_available = FALSE would have silently dropped all 3 million of these rows without any warning, because NULL = FALSE evaluates to NULL in SQL, not TRUE or FALSE. This means: for about a third of this month, I cannot honestly say whether GA4 engagement signals (sessions, scroll rate, engagement rate) are real zeros or simply missing — any feature built from GA4 fields carries this uncertainty unless I explicitly filter with IS TRUE/IS NOT TRUE, as I did.

This also connects to the broader unbalanced panel issue from the data dictionary: only 55 of the warehouse's 104 total clients show any activity in March 2026 at all (from Fact 2), and each client's tracking history starts at a different date (gsc_data_start, ga4_data_start). So this single-month slice is not a fair, uniform sample across all clients — some clients simply aren't represented here yet, and among those that are, GA4 coverage itself is inconsistent within the month.

What this means practically: any feature or claim I make this week is scoped to this one March 2026 partition, for the clients active in it, and GA4-derived features specifically should be treated with caution until I've explicitly filtered for ga4_data_available IS TRUE — which I now know how to do correctly, having seen what breaks when I don't.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.